# AI/BI Adoption Dashboard - Metadata Collection

This notebook collects metadata from various Databricks AI/BI services using the Databricks SDK.

## Tables Created

This notebook creates the following Delta tables:

### Genie
* `adb_genie_spaces` - All Genie spaces
* `adb_genie_conversations` - All conversations within Genie spaces
* `adb_genie_messages` - All messages with query results and errors

### Dashboards
* `adb_dashboards` - All Lakeview dashboards
* `adb_dashboard_schedules` - Dashboard schedules
* `adb_dashboard_subscriptions` - Dashboard subscriptions

### Models & Serving
* `adb_models` - Unity Catalog registered models
* `adb_serving_endpoints` - Model serving endpoints

### Apps
* `adb_apps` - Databricks Apps

## Parameters
* `catalog_name` - Target catalog for tables
* `schema_name` - Target schema for tables

In [0]:
# Uncomment these lines to install the SDK (databricks-sdk>=0.129.0) if running on CLASSIC compute.
# On serverless compute the SDK is provided via the job's serverless environment dependency
# (see deployment_resources/workflows.yml -> environments).
# %pip install 'databricks-sdk>=0.129.0' --upgrade

In [0]:
# Uncomment alongside the %pip line above when running on CLASSIC compute:
# dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog_name", "users")
dbutils.widgets.text("schema_name", "")
dbutils.widgets.text("skip_get_conversations", "false")

skip_get_conversations = dbutils.widgets.get("skip_get_conversations").lower() == 'true'
catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
assert catalog_name and schema_name, "catalog_name and schema_name must be provided"


In [ ]:
from databricks.sdk import WorkspaceClient
from datetime import datetime
import pandas as pd
import json
import pyspark.sql.functions as F
from concurrent.futures import ThreadPoolExecutor, as_completed

w = WorkspaceClient()


def parallel_map(fn, items, max_workers=8):
    """Apply ``fn`` to every item concurrently over a bounded thread pool.

    The metadata ingest is dominated by per-item REST round-trips
    (get_permissions, list_schedules, list_subscriptions, list_conversations,
    list_conversation_messages) that are independent of one another, so we fan
    them out instead of looping serially.

    - Iterable returns from ``fn`` are flattened into the results; scalar
      returns are appended as-is; ``None`` is dropped.
    - Per-item exceptions are collected (not raised) so a single 403/404 does
      not abort the whole batch — mirrors the serial ``try/except: continue``
      the loops used before.
    - ``max_workers`` default 8 is safe for the Permissions API (100/s limit)
      and Lakeview list. Genie list endpoints are throttled tighter, so the
      Genie fan-outs pass a smaller value.
    """
    results, errors = [], []
    if not items:
        return results
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = {pool.submit(fn, item): item for item in items}
        for fut in as_completed(futures):
            try:
                out = fut.result()
            except Exception as exc:  # noqa: BLE001 - collect, don't abort batch
                errors.append(exc)
                continue
            if isinstance(out, (list, tuple)):
                results.extend(out)
            elif out is not None:
                results.append(out)
    if errors:
        print(f"parallel_map: {len(errors)} item(s) failed (continuing). First: {errors[0]}")
    return results


In [ ]:
# List of all tables to drop
table_names = [
    "adb_genie_spaces",
    "adb_genie_conversations",
    "adb_genie_messages",
    "adb_genie_message_statements",
    "adb_genie_message_comments",
    "adb_dashboards",
    "adb_dashboard_schedules",
    "adb_dashboard_subscriptions",
    "adb_models",
    "adb_serving_endpoints",
    "adb_apps"
]

print("Dropping existing tables if they exist...")
for table_name in table_names:
    full_table_name = f"{catalog_name}.{schema_name}.{table_name}"
    try:
        spark.sql(f"DROP TABLE IF EXISTS {full_table_name}")
        print(f"✓ Dropped table: {full_table_name}")
    except Exception as e:
        print(f"⚠ Could not drop {full_table_name}: {e}")

print(f"\nCompleted deletion process for {len(table_names)} tables.")

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, LongType, ArrayType

# Define schemas for all tables
schemas = {
    "adb_genie_spaces": StructType([
        StructField('space_id', StringType(), True),
        StructField('name', StringType(), True),
        StructField('description', StringType(), True),
        StructField('warehouse_id', StringType(), True)
    ]),
    
    "adb_genie_conversations": StructType([
        StructField('space_id', StringType(), True),
        StructField('conversation_id', StringType(), True),
        StructField('created_timestamp', TimestampType(), True),
        StructField('title', StringType(), True),
        StructField('user_id', LongType(), True)
    ]),
    
    "adb_genie_messages": StructType([
        StructField('space_id', StringType(), True),
        StructField('space_name', StringType(), True),
        StructField('message_id', StringType(), True),
        StructField('conversation_id', StringType(), True),
        StructField('user_id', StringType(), True),
        StructField('user_email', StringType(), True),
        StructField('status', StringType(), True),
        StructField('created_timestamp', TimestampType(), True),
        StructField('last_updated_timestamp', TimestampType(), True),
        StructField('user_question', StringType(), True),
        StructField('ai_response', StringType(), True),
        StructField('sql_query', StringType(), True),
        StructField('statement_id', ArrayType(StringType()), True),
        StructField('num_statements', IntegerType(), True),
        StructField('suggested_questions', StringType(), True),
        StructField('num_attachments', IntegerType(), True),
        StructField('feedback_rating', StringType(), True),
        StructField('error_type', StringType(), True),
        StructField('error_message', StringType(), True)
    ]),
    
    "adb_genie_message_statements": StructType([
        StructField('space_id', StringType(), True),
        StructField('space_name', StringType(), True),
        StructField('conversation_id', StringType(), True),
        StructField('message_id', StringType(), True),
        StructField('attachment_id', StringType(), True),
        StructField('attachment_index', IntegerType(), True),
        StructField('statement_id', StringType(), True),
        StructField('sql_query', StringType(), True),
        StructField('query_description', StringType(), True),
        StructField('row_count', LongType(), True),
        StructField('user_id', StringType(), True),
        StructField('user_email', StringType(), True),
        StructField('created_timestamp', TimestampType(), True)
    ]),

    "adb_dashboards": StructType([
        StructField('dashboard_id', StringType(), True),
        StructField('display_name', StringType(), True),
        StructField('create_time', TimestampType(), True),
        StructField('lifecycle_state', StringType(), True),
        StructField('update_time', TimestampType(), True),
        StructField('warehouse_id', StringType(), True)
    ]),
    
    "adb_dashboard_schedules": StructType([
        StructField('dashboard_id', StringType(), True),
        StructField('schedule_id', StringType(), True),
        StructField('create_time', StringType(), True),
        StructField('display_name', StringType(), True),
        StructField('pause_status', StringType(), True)
    ]),
    
    "adb_dashboard_subscriptions": StructType([
        StructField('dashboard_id', StringType(), True),
        StructField('schedule_id', StringType(), True),
        StructField('subscription_id', StringType(), True),
        StructField('create_time', TimestampType(), True),
        StructField('user_id', StringType(), True),
        StructField('destination_id', StringType(), True)
    ]),
    
    "adb_models": StructType([
        StructField('full_name', StringType(), True),
        StructField('name', StringType(), True),
        StructField('catalog_name', StringType(), True),
        StructField('schema_name', StringType(), True),
        StructField('created_at', TimestampType(), True),
        StructField('created_by', StringType(), True),
        StructField('updated_at', TimestampType(), True),
        StructField('updated_by', StringType(), True),
        StructField('owner', StringType(), True),
        StructField('comment', StringType(), True),
        StructField('num_users_with_access', IntegerType(), True),
        StructField('num_groups_with_access', IntegerType(), True)
    ]),
    
    "adb_serving_endpoints": StructType([
        StructField('name', StringType(), True),
        StructField('id', StringType(), True),
        StructField('creation_timestamp', TimestampType(), True),
        StructField('creator', StringType(), True),
        StructField('last_updated_timestamp', TimestampType(), True),
        StructField('state', StringType(), True),
        StructField('models', StringType(), True),
        StructField('num_users_with_access', IntegerType(), True),
        StructField('num_groups_with_access', IntegerType(), True)
    ]),
    
    "adb_apps": StructType([
        StructField('name', StringType(), True),
        StructField('id', StringType(), True),
        StructField('create_time', TimestampType(), True),
        StructField('creator', StringType(), True),
        StructField('update_time', TimestampType(), True),
        StructField('updater', StringType(), True),
        StructField('url', StringType(), True),
        StructField('app_status', StringType(), True),
        StructField('compute_status', StringType(), True),
        StructField('num_users_with_access', IntegerType(), True),
        StructField('num_groups_with_access', IntegerType(), True)
    ])
}

# Create all empty tables
print("Initializing empty tables...")
for table_name, schema in schemas.items():
    full_table_name = f"{catalog_name}.{schema_name}.{table_name}"
    empty_df = spark.createDataFrame([], schema)
    empty_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_table_name)
    print(f"✓ Created empty table: {full_table_name}")

print(f"\nAll {len(schemas)} tables initialized successfully!")

## Genie Spaces, Conversations, and Messages

This section collects metadata about Genie spaces and their usage:
* **adb_genie_spaces**: All Genie spaces in the workspace
* **adb_genie_conversations**: All conversations within each Genie space
* **adb_genie_messages**: All messages within each conversation, including query results and errors

In [0]:
spaces = []
page_token = None

# w = WorkspaceClient()

while True:
    response = w.genie.list_spaces(page_token=page_token)
    for s in response.spaces:
        spaces.append({
            "space_id": getattr(s, "space_id", None),
            "name": getattr(s, "title", None),
            "description": getattr(s, "description", None),
            "warehouse_id": getattr(s, "warehouse_id", None)
        })
    if not response.next_page_token or response.next_page_token == "":
        break
    page_token = response.next_page_token

pdf = pd.DataFrame(spaces)
if not pdf.empty:
    genie_spaces_df = spark.createDataFrame(pdf)
    genie_spaces_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        f"{catalog_name}.{schema_name}.adb_genie_spaces"
    )
    print(f"Loaded {genie_spaces_df.count()} Genie spaces into table adb_genie_spaces")
else:
    print("No Genie spaces found.")

In [ ]:
from databricks.sdk import WorkspaceClient
from datetime import datetime
import pandas as pd

#w = WorkspaceClient()

has_manage_on_all_spaces = True
include_all_conversations = has_manage_on_all_spaces

# Genie list endpoints are throttled tighter than the Permissions API, so keep
# the conversation fan-out modest.
GENIE_MAX_WORKERS = 8

def _fetch_space_conversations(space_id):
    """Fetch ALL conversations for one space (paginated). Worker for parallel_map."""
    out = []
    try:
        page_token = None
        while True:
            response = w.genie.list_conversations(
                space_id=space_id, include_all=include_all_conversations, page_token=page_token
            )
            for conv in response.conversations:
                conv_dict = conv.as_dict()
                conv_dict['space_id'] = space_id
                out.append(conv_dict)
            if not response.next_page_token or response.next_page_token == "":
                break
            page_token = response.next_page_token
    except Exception as e:
        print(f"Error fetching conversations for space {space_id}: {e}")
    return out

conversations = []

# Check if the adb_genie_spaces table exists before proceeding
if not spark.catalog.tableExists(f"{catalog_name}.{schema_name}.adb_genie_spaces"):
    print("No Genie spaces table found. Skipping conversation collection.")
else:
    # Get all space IDs from the previously created table
    space_ids = [
        row.space_id
        for row in spark.table(f"{catalog_name}.{schema_name}.adb_genie_spaces").select('space_id').collect()
    ]

    if not space_ids:
        print("No Genie spaces found in table. Skipping conversation collection.")
    else:
        print(f"Fetching conversations for {len(space_ids)} Genie spaces "
              f"(concurrency={GENIE_MAX_WORKERS})...")
        conversations = parallel_map(
            _fetch_space_conversations, space_ids, max_workers=GENIE_MAX_WORKERS
        )

if conversations:
    genie_conversations_df = spark.createDataFrame(conversations).withColumn('created_timestamp', F.from_unixtime(F.col('created_timestamp')/1000).cast('timestamp'))
    genie_conversations_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        f"{catalog_name}.{schema_name}.adb_genie_conversations"
    )
    print(f"Loaded {genie_conversations_df.count()} Genie conversations into table adb_genie_conversations")
else:
    print("No Genie conversations found.")

In [ ]:
from typing import Optional, List, Dict, Any
from databricks.sdk.service.iam import User
import threading

# The message fetch is a per-conversation network round-trip; independent
# conversations are fanned out over the shared bounded pool (parallel_map /
# GENIE_MAX_WORKERS, defined earlier) instead of a serial loop. This was the
# single biggest wall-clock lever — a serial sweep of a busy workspace could
# take hours.

# Cache for user ID to email mappings (thread-safe: guarded by a lock because
# the message fetch below runs multi-threaded).
USER_CACHE = {}
_USER_CACHE_LOCK = threading.Lock()

def _resolve_user_email(user_id: Optional[str], w: WorkspaceClient) -> Optional[str]:
    """Resolves a user ID to an email address using the SCIM Users API."""
    if not user_id or user_id in ("None", "0"):
        return None

    with _USER_CACHE_LOCK:
        if user_id in USER_CACHE:
            return USER_CACHE[user_id]

    try:
        user: User = w.users.get(user_id)
        email = user.user_name
    except Exception:
        # Return a placeholder if user lookup fails
        email = f"ID_{user_id}"

    with _USER_CACHE_LOCK:
        USER_CACHE[user_id] = email
    return email

def _extract_message_data(message, space_id: str, space_name: str, w: WorkspaceClient) -> Dict[str, Any]:
    """Extracts relevant fields from a GenieMessage into a flat dictionary.

    IMPORTANT (grain): a single Genie message can contain MULTIPLE query
    attachments, each with its own statement_id. Genie runs several SQL
    statements to answer one question in both chat mode (exploratory
    data-sourcing queries) and, especially, agent mode (multi-step research).
    We therefore keep statement_ids as a proper ARRAY and also emit a
    normalised one-row-per-statement list ('statement_records') so downstream
    joins to dbsql_cost_per_query are 1:1 on statement_id. The previous
    implementation ' | '-joined the ids into a single scalar string, which made
    equality joins on statement_id silently miss every multi-statement message.
    """
    msg_dict = message.as_dict()

    # Extract IDs (API has both 'id' and 'message_id' for legacy compatibility)
    msg_id = msg_dict.get('message_id') or msg_dict.get('id')
    user_id = msg_dict.get('user_id')
    user_email = _resolve_user_email(str(user_id) if user_id else None, w)

    record = {
        'space_id': space_id,
        'space_name': space_name,
        'message_id': msg_id,
        'conversation_id': msg_dict.get('conversation_id'),
        'user_id': str(user_id) if user_id else None,
        'user_email': user_email,
        'status': str(msg_dict.get('status')) if msg_dict.get('status') else None,
        'created_timestamp': msg_dict.get('created_timestamp'),
        'last_updated_timestamp': msg_dict.get('last_updated_timestamp'),
        'user_question': msg_dict.get('content'),
    }

    # Process attachments
    ai_responses, sql_queries, statement_ids, suggested_qs = [], [], [], []
    statement_records = []  # one entry per query attachment (statement grain)
    attachments = msg_dict.get('attachments') or []

    for att_index, att in enumerate(attachments):
        attachment_id = att.get('attachment_id')

        # Text attachment
        if text_obj := att.get('text'):
            ai_responses.append(text_obj.get('content', ''))

        # Query attachment. NOTE: 0..N of these per message.
        if query_obj := att.get('query'):
            sql_text = query_obj.get('query', '')
            sql_queries.append(sql_text)
            stmt_id = query_obj.get('statement_id')
            if stmt_id:
                statement_ids.append(str(stmt_id))
            # Emit a normalised statement-grain record even if statement_id is
            # missing (e.g. cached/None), so we can measure that gap too.
            qrm = query_obj.get('query_result_metadata') or {}
            statement_records.append({
                'space_id': space_id,
                'space_name': space_name,
                'conversation_id': msg_dict.get('conversation_id'),
                'message_id': msg_id,
                'attachment_id': attachment_id,
                'attachment_index': att_index,
                'statement_id': str(stmt_id) if stmt_id else None,
                'sql_query': sql_text,
                'query_description': query_obj.get('description'),
                'row_count': qrm.get('row_count'),
                'user_id': str(user_id) if user_id else None,
                'user_email': user_email,
                'created_timestamp': msg_dict.get('created_timestamp'),
            })

        # Suggested questions attachment
        if sq_obj := att.get('suggested_questions'):
            suggested_qs.extend(sq_obj.get('questions', []))

    def join_non_empty(items: List, sep: str = ' | ') -> Optional[str]:
        filtered = list(filter(None, items))
        return sep.join(filtered) if filtered else None

    record.update({
        'ai_response': join_non_empty(ai_responses),
        'sql_query': join_non_empty(sql_queries),
        # statement_id is now a proper ARRAY<STRING>: one element per executed
        # statement. filter(None) drops query attachments that had no id.
        'statement_id': list(filter(None, statement_ids)),
        'num_statements': len(list(filter(None, statement_ids))),
        'suggested_questions': join_non_empty(suggested_qs, ', '),
        'num_attachments': len(attachments),
    })

    # Feedback rating
    feedback = msg_dict.get('feedback') or {}
    record['feedback_rating'] = str(feedback.get('rating')) if feedback.get('rating') else 'NONE'
    record['num_comments'] = 0  # backfilled after comments are fetched (see comments silver cell)

    # Error info
    error = msg_dict.get('error')
    if error and isinstance(error, dict):
        record['error_type'] = error.get('type')
        record['error_message'] = error.get('message') or error.get('error')
    elif error:
        record['error_type'] = 'Unknown'
        record['error_message'] = str(error)
    else:
        record['error_type'] = None
        record['error_message'] = None

    return record, statement_records

def _fetch_conversation_messages(conv_tuple):
    """Fetch + parse ALL messages for one conversation (paginated).

    ``conv_tuple`` is (space_id, space_name, conversation_id). Returns a list of
    (message_record, statement_records) pairs. Worker for parallel_map; catches
    its own errors so one bad conversation doesn't sink the batch.
    """
    space_id, space_name, conversation_id = conv_tuple
    pairs = []
    try:
        page_token = None
        while True:
            response = w.genie.list_conversation_messages(
                space_id=space_id,
                conversation_id=conversation_id,
                page_token=page_token
            )
            if not response.messages:
                break
            for msg in response.messages:
                pairs.append(_extract_message_data(msg, space_id, space_name, w))
            if not response.next_page_token or response.next_page_token == "":
                break
            page_token = response.next_page_token
    except Exception as e:
        print(f"Error fetching messages for conversation {conversation_id} in space {space_id}: {e}")
    return pairs

messages = []
statements = []  # statement-grain rows across all messages

# Check if required tables exist before proceeding
if not spark.catalog.tableExists(f"{catalog_name}.{schema_name}.adb_genie_spaces"):
    print("No Genie spaces table found. Skipping message collection.")
elif not spark.catalog.tableExists(f"{catalog_name}.{schema_name}.adb_genie_conversations"):
    print("No Genie conversations table found. Skipping message collection.")
elif skip_get_conversations:
    print('Skipping conversations pull as skip_get_conversations is set to true')
else:
    # Get all space_id, space_name, and conversation_id from the previously created tables
    conversation_data = [
        (row.space_id, row.space_name, row.conversation_id)
        for row in spark.sql(f"""
            SELECT c.space_id, s.name as space_name, c.conversation_id
            FROM {catalog_name}.{schema_name}.adb_genie_conversations c
            INNER JOIN {catalog_name}.{schema_name}.adb_genie_spaces s
            ON c.space_id = s.space_id
        """).collect()
    ]

    if not conversation_data:
        print("No conversations found in table. Skipping message collection.")
    else:
        print(f"Fetching messages for {len(conversation_data)} conversations "
              f"(concurrency={GENIE_MAX_WORKERS})...")
        # Fan out the per-conversation fetches over the shared bounded pool.
        pairs = parallel_map(
            _fetch_conversation_messages, conversation_data, max_workers=GENIE_MAX_WORKERS
        )
        for message_data, statement_records in pairs:
            messages.append(message_data)
            statements.extend(statement_records)
        print(f"Fetched {len(messages)} messages / {len(statements)} statement rows.")

if messages:
    genie_messages_df = spark.createDataFrame(
        messages,
        'space_id string, space_name string, message_id string, conversation_id string, user_id string, user_email string, status string, created_timestamp bigint, last_updated_timestamp bigint, user_question string, ai_response string, sql_query string, statement_id array<string>, num_statements int, suggested_questions string, num_attachments int, feedback_rating string, num_comments bigint, error_type string, error_message string'
    ) \
        .withColumn('created_timestamp', F.from_unixtime(F.col('created_timestamp')/1000).cast('timestamp')) \
        .withColumn('last_updated_timestamp', F.from_unixtime(F.col('last_updated_timestamp')/1000).cast('timestamp'))
    genie_messages_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        f"{catalog_name}.{schema_name}.adb_genie_messages"
    )
    print(f"Loaded {genie_messages_df.count()} Genie messages into table adb_genie_messages")
else:
    print("No Genie messages found or process skipped.")

# Statement-grain bridge table: one row per executed SQL statement.
# This is the table that joins 1:1 to dbsql_cost_per_query on statement_id,
# enabling true cost-per-message / cost-per-question attribution. Consistent
# across chat mode and agent mode: both surface query attachments here, and a
# multi-statement turn simply produces multiple rows.
if statements:
    genie_statements_df = spark.createDataFrame(
        statements,
        'space_id string, space_name string, conversation_id string, message_id string, attachment_id string, attachment_index int, statement_id string, sql_query string, query_description string, row_count long, user_id string, user_email string, created_timestamp bigint'
    ).withColumn('created_timestamp', F.from_unixtime(F.col('created_timestamp')/1000).cast('timestamp'))
    genie_statements_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        f"{catalog_name}.{schema_name}.adb_genie_message_statements"
    )
    print(f"Loaded {genie_statements_df.count()} Genie statement rows into table adb_genie_message_statements")
else:
    print("No Genie statement rows found or process skipped.")


In [ ]:
# =============================================================================
# Silver 2: Genie message COMMENTS (adb_genie_message_comments)
# =============================================================================
# Comments are a per-message ARRAY from a separate endpoint
# (.../messages/{id}/comments). A message can have 0..N comments, each with its
# own message_comment_id, author and timestamp (the thumbs-down "reason" text
# also lands here). We therefore explode them into a dedicated one-row-per-comment
# silver table -- NEVER pipe-joined into a string (same principle as statement_ids).
#
# This is the source of truth for comment text. The bronze table (adb_genie_messages)
# only carries num_comments (a count) so "messages with vs without comments" is
# answerable there without duplicating the text.
#
# PERFORMANCE (this was the multi-hour bottleneck):
#   1. We ONLY call /comments for messages that actually have feedback
#      (feedback_rating != 'NONE'). A comment / thumbs-down reason cannot exist
#      without a rating, so unrated messages are guaranteed to have no comments
#      and calling /comments for them is pure waste. On a typical workspace the
#      vast majority of messages are unrated, so this alone removes ~90%+ of the
#      calls with zero loss of coverage.
#   2. The remaining calls are fanned out over the shared bounded pool
#      (parallel_map / GENIE_MAX_WORKERS) instead of a serial loop.
# Set enable_genie_feedback_comments=false to skip comment fetching entirely.

dbutils.widgets.text("enable_genie_feedback_comments", "true")
enable_feedback_comments = dbutils.widgets.get("enable_genie_feedback_comments").lower() == "true"

from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType

def _fetch_comments(space_id, conversation_id, message_id):
    """Return the raw list of comment dicts for a message (best-effort)."""
    path = f"/api/2.0/genie/spaces/{space_id}/conversations/{conversation_id}/messages/{message_id}/comments"
    try:
        resp = w.api_client.do("GET", path)
        return (resp or {}).get("comments", []) or []
    except Exception:
        try:
            import requests as _rq
            _headers = w.config.authenticate()
            _r = _rq.get(f"{w.config.host}{path}", headers=_headers, timeout=30)
            if _r.ok:
                return (_r.json() or {}).get("comments", []) or []
        except Exception:
            pass
    return []

def _fetch_comments_for_message(m):
    """Worker: fetch + normalise all comments for one message dict `m`.

    Returns a list of comment row dicts (possibly empty). Worker for
    parallel_map; only called for messages already known to carry feedback.
    """
    message_id = m.get("message_id")
    conversation_id = m.get("conversation_id")
    space_id = m.get("space_id")
    if not (message_id and conversation_id and space_id):
        return []
    rows = []
    for c in _fetch_comments(space_id, conversation_id, message_id):
        content = (c.get("content") or "").strip()
        if not content:
            continue
        rows.append({
            "message_comment_id": c.get("message_comment_id"),
            "space_id": c.get("space_id") or space_id,
            "space_name": m.get("space_name"),
            "conversation_id": c.get("conversation_id") or conversation_id,
            "message_id": c.get("message_id") or message_id,
            "user_id": str(c.get("user_id")) if c.get("user_id") is not None else None,
            "comment_text": content,
            "created_timestamp": c.get("created_timestamp"),
        })
    return rows

comment_rows = []
# message_id -> number of comments, fed back onto the bronze table below
comment_counts = {}

if enable_feedback_comments and messages:
    # Only messages that have feedback can have comments / thumbs-down reasons.
    # De-dup on message_id while we're at it.
    seen = set()
    rated_messages = []
    for m in messages:
        mid = m.get("message_id")
        if not mid or mid in seen:
            continue
        seen.add(mid)
        if (m.get("feedback_rating") or "NONE") != "NONE":
            rated_messages.append(m)

    print(f"Fetching comments for {len(rated_messages)} rated messages "
          f"(of {len(seen)} total; concurrency={GENIE_MAX_WORKERS})...")

    comment_rows = parallel_map(
        _fetch_comments_for_message, rated_messages, max_workers=GENIE_MAX_WORKERS
    )
    for row in comment_rows:
        mid = row["message_id"]
        comment_counts[mid] = comment_counts.get(mid, 0) + 1

comments_schema = "message_comment_id string, space_id string, space_name string, conversation_id string, message_id string, user_id string, comment_text string, created_timestamp bigint"

if comment_rows:
    comments_df = spark.createDataFrame(comment_rows, comments_schema) \
        .withColumn("created_timestamp", F.from_unixtime(F.col("created_timestamp") / 1000).cast("timestamp"))
    comments_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        f"{catalog_name}.{schema_name}.adb_genie_message_comments"
    )
    print(f"Loaded {comments_df.count()} comment rows into adb_genie_message_comments")
else:
    spark.createDataFrame([], StructType([
        StructField("message_comment_id", StringType(), True),
        StructField("space_id", StringType(), True),
        StructField("space_name", StringType(), True),
        StructField("conversation_id", StringType(), True),
        StructField("message_id", StringType(), True),
        StructField("user_id", StringType(), True),
        StructField("comment_text", StringType(), True),
        StructField("created_timestamp", TimestampType(), True),
    ])).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        f"{catalog_name}.{schema_name}.adb_genie_message_comments"
    )
    print("No Genie comments found; created empty adb_genie_message_comments table")

# ---- Backfill num_comments onto the bronze adb_genie_messages table ----
# Bronze already exists (written in the previous cell). We add the per-message
# comment count so "messages with vs without comments" is answerable on bronze.
if comment_counts:
    counts_df = spark.createDataFrame(
        [{"message_id": k, "num_comments": v} for k, v in comment_counts.items()],
        "message_id string, num_comments bigint",
    )
    bronze = spark.table(f"{catalog_name}.{schema_name}.adb_genie_messages")
    updated = bronze.drop("num_comments").join(counts_df, on="message_id", how="left") \
        .withColumn("num_comments", F.coalesce(F.col("num_comments"), F.lit(0)).cast("bigint"))
    updated.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        f"{catalog_name}.{schema_name}.adb_genie_messages"
    )
    print("Backfilled num_comments onto adb_genie_messages")


## Lakeview Dashboards, Schedules, and Subscriptions

This section collects metadata about Lakeview dashboards and their distribution:
* **adb_dashboards**: All Lakeview dashboards in the workspace
* **adb_dashboard_schedules**: Scheduled refresh/distribution for dashboards
* **abd_dashboard_subscriptions**: User and destination subscriptions for dashboard schedules

In [0]:
from databricks.sdk import WorkspaceClient
from datetime import datetime

#w = WorkspaceClient()

rows = []
for d in w.lakeview.list(page_size=100):
    d_dict = d.as_dict()
    # Convert timestamps to Python datetime if they are not already
    create_time = d_dict.get("create_time")
    update_time = d_dict.get("update_time")
    if isinstance(create_time, str):
        try:
            create_time = datetime.fromisoformat(create_time)
        except Exception:
            create_time = None
    if isinstance(update_time, str):
        try:
            update_time = datetime.fromisoformat(update_time)
        except Exception:
            update_time = None
    rows.append((
        d_dict.get("dashboard_id"),
        d_dict.get("display_name"),
        create_time,
        d_dict.get("lifecycle_state"),
        update_time,
        d_dict.get("warehouse_id")
    ))

from pyspark.sql.types import StructType, StructField, StringType, TimestampType
schema = StructType([
    StructField('dashboard_id', StringType(), True),
    StructField('display_name', StringType(), True),
    StructField('create_time', TimestampType(), True),
    StructField('lifecycle_state', StringType(), True),
    StructField('update_time', TimestampType(), True),
    StructField('warehouse_id', StringType(), True)
])

df = spark.createDataFrame(
    rows,
    schema
)
df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(f"{catalog_name}.{schema_name}.adb_dashboards")
#display(df)

In [ ]:
from databricks.sdk import WorkspaceClient
from datetime import datetime

dashboard_ids = [
    row.dashboard_id
    for row in spark.table(f"{catalog_name}.{schema_name}.adb_dashboards").select("dashboard_id").collect()
]

def _fetch_schedules(dashboard_id):
    """Fetch all schedules for one dashboard. Worker for parallel_map."""
    out = []
    for sched in w.lakeview.list_schedules(dashboard_id=dashboard_id):
        sched_dict = sched.as_dict()
        out.append({
            "dashboard_id": dashboard_id,
            "schedule_id": sched_dict.get("schedule_id"),
            "create_time": sched_dict.get("create_time"),
            "display_name": sched_dict.get("display_name"),
            "pause_status": sched_dict.get("pause_status")
        })
    return out

# Fan out the per-dashboard schedule fetches (independent REST calls).
schedules = parallel_map(_fetch_schedules, dashboard_ids, max_workers=8)

if schedules:
    import pandas as pd
    pdf_sched = pd.DataFrame(schedules)
    spark_df_sched = spark.createDataFrame(pdf_sched)
    spark.sql(f"DROP TABLE IF EXISTS {catalog_name}.{schema_name}.adb_dashboard_schedules")
    spark_df_sched.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog_name}.{schema_name}.adb_dashboard_schedules")
    #display(spark_df_sched)

In [ ]:
# Databricks notebook Python
from databricks.sdk import WorkspaceClient
import pandas as pd

source_table = f"{catalog_name}.{schema_name}.adb_dashboards"
schedule_table = f"{catalog_name}.{schema_name}.adb_dashboard_schedules"
target_table = f"{catalog_name}.{schema_name}.adb_dashboard_subscriptions"

# Check if source tables exist
if not spark.catalog.tableExists(schedule_table):
    print(f"Table {schedule_table} not found. Skipping subscription extraction.")
else:
    #w = WorkspaceClient()

    # ---------- helpers ----------
    def get_field(obj_or_dict, *names):
        """Return the first non-None among possible field names (works for dicts or objects)."""
        for n in names:
            if isinstance(obj_or_dict, dict):
                if n in obj_or_dict and obj_or_dict[n] is not None:
                    return obj_or_dict[n]
            else:
                v = getattr(obj_or_dict, n, None)
                if v is not None:
                    return v
        return None

    def to_dict_safe(x):
        return (getattr(x, "as_dict", lambda: {})() or {}) if x is not None else {}

    def _fetch_subscriptions(sched_row):
        """Fetch all subscriptions for one (dashboard_id, schedule_id). Worker for parallel_map."""
        dashboard_id = sched_row["dashboard_id"]
        schedule_id = sched_row["schedule_id"]
        out = []
        for sub in w.lakeview.list_subscriptions(dashboard_id=dashboard_id, schedule_id=schedule_id):
            subdict = to_dict_safe(sub)
            subscriber = get_field(subdict, "subscriber") or get_field(sub, "subscriber") or {}

            user_subscriber = get_field(subscriber, "user_subscriber")
            destination_subscriber = get_field(subscriber, "destination_subscriber")

            user_id = (
                get_field(user_subscriber or {}, "user_id", "id")
                if user_subscriber is not None else None
            )

            subscription_id = get_field(subdict, "subscription_id") or get_field(sub, "subscription_id")
            create_time = get_field(subdict, "create_time") or get_field(sub, "create_time")

            # capture destination info if present
            destination_id = get_field(destination_subscriber or {}, "destination_id", "destination", "id")

            out.append({
                "dashboard_id": dashboard_id,
                "schedule_id": schedule_id,
                "subscription_id": subscription_id,
                "create_time": create_time,
                "user_id": user_id,
                "destination_id": destination_id,
            })
        return out

    # ---------- fan out over (dashboard, schedule) pairs ----------
    sched_df = spark.table(schedule_table).collect()
    rows = parallel_map(_fetch_subscriptions, sched_df, max_workers=8)
    print(f"Collected {len(rows)} subscription rows across dashboards.")

    # ---------- write to Delta table ----------
    pdf = pd.DataFrame(rows)

    if pdf.empty:
        spark_df = spark.createDataFrame([], schemas["adb_dashboard_subscriptions"])
    else:
        spark_df = spark.createDataFrame(pdf)

    spark.sql(f"DROP TABLE IF EXISTS {target_table}")

    from pyspark.sql import functions as F
    spark_df = (
        spark_df
        .withColumn("create_time", F.to_timestamp("create_time"))
        .select("dashboard_id", "schedule_id", "subscription_id", "create_time", "user_id", "destination_id")
    )

    spark_df.write.format("delta").mode("overwrite").saveAsTable(target_table)
    print(f"Successfully created {target_table}")

## Unity Catalog Models

This section collects metadata about registered models in Unity Catalog:
* **adb_models**: All registered models with permissions and metadata

In [ ]:
from databricks.sdk import WorkspaceClient
from datetime import datetime
import pandas as pd

#w = WorkspaceClient()

def _build_model_row(model):
    """Fetch permissions for one model and build its row. Worker for parallel_map."""
    model_dict = model.as_dict() if hasattr(model, 'as_dict') else {}

    # Get permissions summary (the per-item REST call we fan out)
    num_users_with_access = 0
    num_groups_with_access = 0
    try:
        full_name = model_dict.get('full_name') or f"{model_dict.get('catalog_name', '')}.{model_dict.get('schema_name', '')}.{model_dict.get('name', '')}"
        if full_name and full_name != '..':
            perms = w.registered_models.get_permissions(full_name)
            if perms and hasattr(perms, 'access_control_list'):
                acl = perms.access_control_list or []
                for entry in acl:
                    if hasattr(entry, 'user_name') and entry.user_name:
                        num_users_with_access += 1
                    if hasattr(entry, 'group_name') and entry.group_name:
                        num_groups_with_access += 1
    except Exception:
        # Permissions may not be accessible, continue without them
        pass

    # Convert timestamps
    created_at = model_dict.get('created_at')
    updated_at = model_dict.get('updated_at')
    if isinstance(created_at, (int, float)) and created_at > 0:
        try:
            created_at = datetime.fromtimestamp(created_at / 1000)
        except Exception:
            created_at = None
    if isinstance(updated_at, (int, float)) and updated_at > 0:
        try:
            updated_at = datetime.fromtimestamp(updated_at / 1000)
        except Exception:
            updated_at = None

    return {
        "full_name": model_dict.get('full_name'),
        "name": model_dict.get('name'),
        "catalog_name": model_dict.get('catalog_name'),
        "schema_name": model_dict.get('schema_name'),
        "created_at": created_at,
        "created_by": model_dict.get('created_by'),
        "updated_at": updated_at,
        "updated_by": model_dict.get('updated_by'),
        "owner": model_dict.get('owner'),
        "comment": model_dict.get('comment'),
        "num_users_with_access": num_users_with_access,
        "num_groups_with_access": num_groups_with_access
    }

models = []
try:
    # Materialize the (paginated) list first, then fan out the per-model
    # permission lookups over the bounded pool.
    model_objs = list(w.registered_models.list())
    models = parallel_map(_build_model_row, model_objs, max_workers=8)
except Exception as e:
    print(f"Error listing models: {e}")

pdf = pd.DataFrame(models)
if not pdf.empty:
    spark_df = spark.createDataFrame(pdf)
    spark_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        f"{catalog_name}.{schema_name}.adb_models"
    )
    print(f"Loaded {spark_df.count()} models into table adb_models")
else:
    print("No models found.")

## Model Serving Endpoints

This section collects metadata about model serving endpoints:
* **adb_serving_endpoints**: All serving endpoints with their models and permissions

In [ ]:
from databricks.sdk import WorkspaceClient
from datetime import datetime
import pandas as pd

#w = WorkspaceClient()

def _build_endpoint_row(endpoint):
    """Fetch permissions for one serving endpoint and build its row. Worker for parallel_map."""
    endpoint_dict = endpoint.as_dict() if hasattr(endpoint, 'as_dict') else {}

    # Get permissions summary (the per-item REST call we fan out)
    num_users_with_access = 0
    num_groups_with_access = 0
    try:
        endpoint_name = endpoint_dict.get('name')
        if endpoint_name:
            perms = w.serving_endpoints.get_permissions(endpoint_name)
            if perms and hasattr(perms, 'access_control_list'):
                acl = perms.access_control_list or []
                for entry in acl:
                    if hasattr(entry, 'user_name') and entry.user_name:
                        num_users_with_access += 1
                    if hasattr(entry, 'group_name') and entry.group_name:
                        num_groups_with_access += 1
    except Exception:
        # Permissions may not be accessible, continue without them
        pass

    # Extract model info from config
    config = endpoint_dict.get('config', {})
    models_info = []
    if isinstance(config, dict):
        served_models = config.get('served_models', [])
        if isinstance(served_models, list):
            for model in served_models:
                if isinstance(model, dict):
                    model_name = model.get('model_name') or model.get('name')
                    if model_name:
                        models_info.append(model_name)

    # Convert timestamps
    creation_timestamp = endpoint_dict.get('creation_timestamp')
    last_updated_timestamp = endpoint_dict.get('last_updated_timestamp')
    if isinstance(creation_timestamp, (int, float)) and creation_timestamp > 0:
        try:
            creation_timestamp = datetime.fromtimestamp(creation_timestamp / 1000)
        except Exception:
            creation_timestamp = None
    if isinstance(last_updated_timestamp, (int, float)) and last_updated_timestamp > 0:
        try:
            last_updated_timestamp = datetime.fromtimestamp(last_updated_timestamp / 1000)
        except Exception:
            last_updated_timestamp = None

    return {
        "name": endpoint_dict.get('name'),
        "id": endpoint_dict.get('id'),
        "creation_timestamp": creation_timestamp,
        "creator": endpoint_dict.get('creator'),
        "last_updated_timestamp": last_updated_timestamp,
        "state": endpoint_dict.get('state'),
        "models": ','.join(models_info) if models_info else None,
        "num_users_with_access": num_users_with_access,
        "num_groups_with_access": num_groups_with_access
    }

serving_endpoints = []
try:
    endpoint_objs = list(w.serving_endpoints.list())
    serving_endpoints = parallel_map(_build_endpoint_row, endpoint_objs, max_workers=8)
except Exception as e:
    print(f"Error listing serving endpoints: {e}")

pdf = pd.DataFrame(serving_endpoints)
if not pdf.empty:
    spark_df = spark.createDataFrame(pdf)
    spark_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        f"{catalog_name}.{schema_name}.adb_serving_endpoints"
    )
    print(f"Loaded {spark_df.count()} serving endpoints into table adb_serving_endpoints")
else:
    print("No serving endpoints found.")


## Databricks Apps

This section collects metadata about Databricks Apps:
* **adb_apps**: All apps with their status and permissions

In [ ]:
from databricks.sdk import WorkspaceClient
from datetime import datetime
import pandas as pd

#w = WorkspaceClient()

def _build_app_row(app):
    """Fetch permissions for one app and build its row. Worker for parallel_map."""
    app_dict = app.as_dict() if hasattr(app, 'as_dict') else {}

    # Get permissions summary (the per-item REST call we fan out)
    num_users_with_access = 0
    num_groups_with_access = 0
    try:
        app_name = app_dict.get('name')
        if app_name:
            perms = w.apps.get_permissions(app_name)
            if perms and hasattr(perms, 'access_control_list'):
                acl = perms.access_control_list or []
                for entry in acl:
                    if hasattr(entry, 'user_name') and entry.user_name:
                        num_users_with_access += 1
                    if hasattr(entry, 'group_name') and entry.group_name:
                        num_groups_with_access += 1
    except Exception:
        # Permissions may not be accessible, continue without them
        pass

    # Convert timestamps
    create_time = app_dict.get('create_time')
    update_time = app_dict.get('update_time')
    if isinstance(create_time, str):
        try:
            create_time = datetime.fromisoformat(create_time.replace('Z', '+00:00'))
        except Exception:
            create_time = None
    if isinstance(update_time, str):
        try:
            update_time = datetime.fromisoformat(update_time.replace('Z', '+00:00'))
        except Exception:
            update_time = None

    # Status: the Apps SDK object has NO flat 'app_status' key; status lives in nested
    # structs. Reading the flat keys was the bug that left adb_apps empty: app_status came
    # back as an all-NULL column and compute_status came back as a raw dict ({'state','message'}),
    # and a dict / all-NULL object column breaks spark.createDataFrame() inference. Flatten
    # both to scalar strings (matching the adb_apps schema, both StringType):
    #   compute_status <- compute_status.state          (ACTIVE / STOPPED / ERROR / ...)
    #   app_status     <- active_deployment.status.state (SUCCEEDED / IN_PROGRESS / ...; the
    #                     app's latest deployment state, NULL when the app was never deployed)
    compute_status = (app_dict.get('compute_status') or {}).get('state')
    app_status = (((app_dict.get('active_deployment') or {}).get('status')) or {}).get('state')

    return {
        "name": app_dict.get('name'),
        "id": app_dict.get('id'),
        "create_time": create_time,
        "creator": app_dict.get('creator'),
        "update_time": update_time,
        "updater": app_dict.get('updater'),
        "url": app_dict.get('url'),
        "app_status": app_status,
        "compute_status": compute_status,
        "num_users_with_access": num_users_with_access,
        "num_groups_with_access": num_groups_with_access
    }

apps = []
try:
    app_objs = list(w.apps.list())
    apps = parallel_map(_build_app_row, app_objs, max_workers=8)
except Exception as e:
    print(f"Error listing apps: {e}")

pdf = pd.DataFrame(apps)
if not pdf.empty:
    spark_df = spark.createDataFrame(pdf)
    spark_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        f"{catalog_name}.{schema_name}.adb_apps"
    )
    print(f"Loaded {spark_df.count()} apps into table adb_apps")
else:
    print("No apps found.")
